In [ ]:
!pip install --no-deps spacy-transformers 'spacy_alignments'
!python -m spacy download en_core_web_trf
!pip install -q "transformers==4.30.0" "tokenizers==0.13.3" "huggingface_hub<1.0"

import spacy
import torch
import cupy
import sys
import numpy as np
import scipy
import scipy.special
from spacy.training import Example
from spacy.scorer import Scorer
from spacy.tokens import DocBin
from spacy.util import filter_spans
from tqdm import tqdm
import json
import warnings
import cupy as cp
    
warnings.filterwarnings('ignore')

print(f"spaCy version: {spacy.__version__}")
print(f"PyTorch version: {torch.__version__}")
print(f"CuPy version: {cupy.__version__}")

In [ ]:
import transformers.tokenization_utils as _tu
from transformers.tokenization_utils_base import BatchEncoding

if not hasattr(_tu, "BatchEncoding"):
    _tu.BatchEncoding = BatchEncoding

try:
    from transformers.file_utils import ModelOutput
except ImportError:
    import transformers.utils as _utils
    import transformers.file_utils as _file_utils_shim 

In [ ]:
import transformers.tokenization_utils as _tu
from transformers.tokenization_utils_base import BatchEncoding
_tu.BatchEncoding = BatchEncoding

import spacy
nlp = spacy.load("/kaggle/input/datasets/.../model/model-best")

In [ ]:
# Experiment 1: Comparison between the time taken by the LLM + NER cascade system to label PHI and the time taken by the NER system alone.

import time

import pandas as pd
import spacy
import torch
from transformers import AutoTokenizer, T5ForSequenceClassification


CSV_PATH = "/kaggle/input/datasets/.../Dataset_test.csv"

TEXT_COLUMN = "final_text"

LLM_BASE_CHECKPOINT = "hossboll/clinical-t5"
LLM_WEIGHTS_PATH = "/kaggle/input/datasets/.../ClinicalT5-base_model_tte.pt"

NER_MODEL_PATH = "/kaggle/input/datasets/.../model/model-best"

MAX_INPUT_LENGTH = 256
LLM_BATCH_SIZE = 16
NER_BATCH_SIZE = 32
POSITIVE_CLASS_INDEX = 1

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def sync():
    if DEVICE == "cuda":
        torch.cuda.synchronize()


def load_llm():
    tokenizer = AutoTokenizer.from_pretrained(
        LLM_BASE_CHECKPOINT
    )

    model = T5ForSequenceClassification.from_pretrained(
        LLM_BASE_CHECKPOINT,
        num_labels=2
    )

    state_dict = torch.load(
        LLM_WEIGHTS_PATH,
        map_location=DEVICE
    )

    if isinstance(state_dict, dict) and "model_state_dict" in state_dict:
        state_dict = state_dict["model_state_dict"]

    model.load_state_dict(
        state_dict,
        strict=False
    )

    return tokenizer, model.to(DEVICE).eval()


def load_ner():
    return spacy.load(NER_MODEL_PATH)


@torch.no_grad()
def llm_batch(texts, tokenizer, model):
    inputs = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=MAX_INPUT_LENGTH,
        return_tensors="pt"
    ).to(DEVICE)

    preds = model(**inputs).logits.argmax(dim=-1)

    return [
        "Yes" if p.item() == POSITIVE_CLASS_INDEX else "No"
        for p in preds
    ]


def llm_inference(texts, tokenizer, model):
    sync()
    start = time.perf_counter()

    predictions = []

    for i in range(0, len(texts), LLM_BATCH_SIZE):
        predictions.extend(
            llm_batch(
                texts[i:i + LLM_BATCH_SIZE],
                tokenizer,
                model
            )
        )

    sync()

    return predictions, time.perf_counter() - start


def ner_inference(texts, nlp):
    sync()
    start = time.perf_counter()

    for _ in nlp.pipe(
        texts,
        batch_size=NER_BATCH_SIZE,
        n_process=1
    ):
        pass

    sync()

    return time.perf_counter() - start


def warmup_llm(texts, tokenizer, model):
    if texts:
        llm_batch(
            texts[:LLM_BATCH_SIZE],
            tokenizer,
            model
        )
        sync()


def warmup_ner(texts, nlp):
    if texts:
        list(
            nlp.pipe(
                texts[:NER_BATCH_SIZE],
                batch_size=NER_BATCH_SIZE,
                n_process=1
            )
        )


def main():
    df = pd.read_csv(CSV_PATH)

    if TEXT_COLUMN not in df.columns:
        raise ValueError(
            f"La colonna '{TEXT_COLUMN}' non esiste nel dataset."
        )

    texts = df[TEXT_COLUMN].fillna("").astype(str).tolist()

    tokenizer, llm_model = load_llm()
    nlp = load_ner()

    warmup_llm(texts, tokenizer, llm_model)
    warmup_ner(texts, nlp)

    llm_predictions, llm_time = llm_inference(
        texts,
        tokenizer,
        llm_model
    )

    llm_positive_texts = [
        text
        for text, prediction in zip(texts, llm_predictions)
        if prediction == "Yes"
    ]

    ner_subset_time = ner_inference(
        llm_positive_texts,
        nlp
    )

    strategy_1_time = llm_time + ner_subset_time

    strategy_2_time = ner_inference(
        texts,
        nlp
    )

    if strategy_1_time < strategy_2_time:
        faster_strategy = "LLM → NER subset"
        speedup = strategy_2_time / strategy_1_time
    else:
        faster_strategy = "NER full dataset"
        speedup = strategy_1_time / strategy_2_time

    print(
        f"LLM → NER subset: {strategy_1_time:.6f} s"
    )

    print(
        f"NER full dataset: {strategy_2_time:.6f} s"
    )

    print(
        f"Strategia più veloce: {faster_strategy} "
        f"({speedup:.2f}x più veloce)"
    )


if __name__ == "__main__":
    main()

In [ ]:
# Experiment 2: The LLM analyzes the entire dataset as input and classifies sentences as either containing PHI or not. The NER analyzes only the sentences that the LLM classified as NOT containing PHI, in order to identify and eliminate potential false negatives (FN).

import ast
import json
import re

import pandas as pd
import spacy
import torch
from spacy.training import Example
from transformers import AutoTokenizer, T5ForSequenceClassification


CSV_PATH = "/kaggle/input/datasets/..../Dataset_test.csv"
NER_MODEL_PATH = "/kaggle/input/datasets/.../model/model-best"
LLM_BASE_CHECKPOINT = "hossboll/clinical-t5"
LLM_WEIGHTS_PATH = "/kaggle/input/datasets/.../ClinicalT5-base_model_tte.pt"

TEXT_COLUMN = "final_text"
SUBS_COLUMN = "substitutions_dictionary"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MAX_INPUT_LENGTH = 256
LLM_BATCH_SIZE = 16
POSITIVE_CLASS_INDEX = 1

PLACEHOLDER_TO_LABEL = {
    "<author_clinical_condition>": "CLIN_COND",
    "<author_medical_report>": "MED_REP",
    "<author_genetic>": "GENETIC",
    "<author_fertility>": "FERTILITY",
    "<author_disability>": "DISABILITY",
    "<author_addiction>": "ADDICTION",
}

LABELS = {
    str(k).strip().strip("<>").strip(): v
    for k, v in PLACEHOLDER_TO_LABEL.items()
}

# Converts the content of the substitutions_dictionary column into a real Python dictionary.
def parse_substitutions(value):
    if pd.isna(value):
        return {}

    if isinstance(value, dict):
        return value

    value = str(value).strip()

    if not value or value.lower() in ("nan", "none", "{}"):
        return {}

    try:
        result = ast.literal_eval(value)
        if isinstance(result, dict):
            return result
    except Exception:
        pass

    try:
        result = json.loads(value)
        if isinstance(result, dict):
            return result
    except Exception:
        pass

    try:
        value = (
            value.replace("\u2018", "'")
                 .replace("\u2019", "'")
                 .replace("\u201c", '"')
                 .replace("\u201d", '"')
        )

        if value.startswith("{") and "'" in value and '"' not in value:
            value = value.replace("'", '"')

        result = json.loads(value)
        return result if isinstance(result, dict) else {}
    except Exception:
        return {}


# Count the sentences containing PHI in the input CSV (ground_truth).
def has_phi_ground_truth(value):
    return any(
        str(k).strip().strip("<>").strip() in LABELS
        for k in parse_substitutions(value)
    )


# Load models (NER + LLM)
def load_models():
    nlp = spacy.load(NER_MODEL_PATH)

    tokenizer = AutoTokenizer.from_pretrained(
        LLM_BASE_CHECKPOINT
    )

    model = T5ForSequenceClassification.from_pretrained(
        LLM_BASE_CHECKPOINT,
        num_labels=2
    )

    state_dict = torch.load(
        LLM_WEIGHTS_PATH,
        map_location=DEVICE
    )

    if isinstance(state_dict, dict) and "model_state_dict" in state_dict:
        state_dict = state_dict["model_state_dict"]

    model.load_state_dict(state_dict, strict=False)
    model.to(DEVICE).eval()

    return nlp, tokenizer, model


# LLM prediction
@torch.no_grad()
def predict_phi(texts, tokenizer, model):
    inputs = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=MAX_INPUT_LENGTH,
        return_tensors="pt"
    ).to(DEVICE)

    preds = model(**inputs).logits.argmax(dim=-1)

    return [
        "Yes" if p.item() == POSITIVE_CLASS_INDEX else "No"
        for p in preds
    ]

# Finds substitution values in the text and assigns BIO labels to each token.
def tokenize_and_label(text, substitutions):
    char_labels = ["O"] * len(text)
    text_lower = text.lower()

    for placeholder, value in substitutions.items():
        label = LABELS.get(
            str(placeholder).strip().strip("<>").strip()
        )

        if not label:
            continue

        value = str(value)

        if not value.strip():
            continue

        value_lower = value.lower()
        start = 0

        while True:
            start = text_lower.find(value_lower, start)

            if start < 0:
                break

            for i in range(start, start + len(value)):
                char_labels[i] = "I-" + label

            char_labels[start] = "B-" + label
            start += len(value)

    tokens = []
    tags = []

    for match in re.finditer(r"\S+", text):
        token = match.group()
        start = match.start()
        tag = char_labels[start]

        if tag == "O":
            for i in range(start, match.end()):
                if char_labels[i].startswith("B-"):
                    tag = char_labels[i]
                    break

        if tag.startswith("I-") and (
            start == 0 or char_labels[start - 1] != tag
        ):
            tag = "B-" + tag.split("-", 1)[1]

        if len(token) > 1 and token[-1] in "?.!:;,()[]'":
            token = token[:-1]

        if token:
            tokens.append(token)
            tags.append(tag)

    return tokens, tags


# Build datasets to feed into the NER system
def build_ner_dataset(df):
    records = []

    for _, row in df.iterrows():
        text = row[TEXT_COLUMN]
        text = text if isinstance(text, str) else ""

        tokens, tags = tokenize_and_label(
            text,
            parse_substitutions(row[SUBS_COLUMN])
        )

        records.append({
            "tokens": tokens,
            "ent_tags": tags
        })

    return records


# Convert the dataset prepared for NER into the Example format required by spaCy to evaluate the NER model.
def create_examples(records, nlp):
    examples = []

    for item in records:
        tokens = item["tokens"]
        tags = item["ent_tags"]
        text = " ".join(tokens)

        entities = []
        position = 0

        for token, tag in zip(tokens, tags):
            end = position + len(token)

            if tag != "O":
                entities.append(
                    (position, end, tag)
                )

            position = end + 1

        example = Example.from_dict(
            nlp.make_doc(text),
            {"entities": entities}
        )

        example.predicted = nlp(text)
        examples.append(example)

    return examples


def main():
    nlp, tokenizer, llm_model = load_models()
    df = pd.read_csv(CSV_PATH)

    if TEXT_COLUMN not in df.columns or SUBS_COLUMN not in df.columns:
        raise ValueError("Colonne richieste non presenti nel CSV.")

    df["has_phi_gt"] = df[SUBS_COLUMN].apply(
        has_phi_ground_truth
    )

    texts = df[TEXT_COLUMN].fillna("").astype(str).tolist()
    llm_predictions = []

    for i in range(0, len(texts), LLM_BATCH_SIZE):
        llm_predictions.extend(
            predict_phi(
                texts[i:i + LLM_BATCH_SIZE],
                tokenizer,
                llm_model
            )
        )

    df["llm_pred"] = llm_predictions

    df_nophi = df[
        df["llm_pred"] == "No"
    ].copy()

    ner_records = build_ner_dataset(df_nophi)
    examples = create_examples(ner_records, nlp)

    df_nophi["ner_pred"] = [
        "Yes" if example.predicted.ents else "No"
        for example in examples
    ]

    initial_fn = (
        df_nophi["has_phi_gt"] &
        df_nophi["llm_pred"].eq("No")
    )

    recovered_fn = (
        initial_fn &
        df_nophi["ner_pred"].eq("Yes")
    )

    remaining_fn = (
        initial_fn &
        df_nophi["ner_pred"].eq("No")
    )

    recovery_rate = (
        recovered_fn.sum() / initial_fn.sum()
        if initial_fn.sum() else 0.0
    )

    df["ner_pred"] = "Not evaluated"

    df.loc[
        df["llm_pred"] == "No",
        "ner_pred"
    ] = df_nophi["ner_pred"].values

    df["final_pred"] = "No"

    df.loc[
        df["llm_pred"] == "Yes",
        "final_pred"
    ] = "Yes"

    df.loc[
        (df["llm_pred"] == "No") &
        (df["ner_pred"] == "Yes"),
        "final_pred"
    ] = "Yes"

    print(
        f"\nLLM FN trovati: {initial_fn.sum()}"
    )

    print(
        f"LLM FN eliminati dal NER: "
        f"{recovered_fn.sum()}/{initial_fn.sum()}"
    )

    print(
        f"LLM FN rimanenti: "
        f"{remaining_fn.sum()}"
    )

if __name__ == "__main__":
    main()


In [ ]:
# Experiment 3: The LLM analyzes the entire dataset as input and classifies sentences as either containing PHI or not. The NER analyzes only the sentences that the LLM classified as containing PHI, in order to identify and eliminate potential false positives (FP).

import os
import ast
import json
import re

import pandas as pd
import spacy
import torch
from spacy.scorer import Scorer
from spacy.training import Example
from transformers import AutoTokenizer, T5ForSequenceClassification


CSV_PATH = "/kaggle/input/datasets/..../Dataset_test.csv"
NER_MODEL_PATH = "/kaggle/input/datasets/.../model/model-best"
LLM_BASE_CHECKPOINT = "hossboll/clinical-t5"
LLM_WEIGHTS_PATH = "/kaggle/input/datasets/.../ClinicalT5-base_model_tte.pt"


TEXT_COLUMN = "final_text"
SUBS_COLUMN = "substitutions_dictionary"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MAX_INPUT_LENGTH = 256
LLM_BATCH_SIZE = 16
POSITIVE_CLASS_INDEX = 1

PLACEHOLDER_TO_LABEL = {
    "<author_clinical_condition>": "CLIN_COND",
    "<author_medical_report>": "MED_REP",
    "<author_genetic>": "GENETIC",
    "<author_fertility>": "FERTILITY",
    "<author_disability>": "DISABILITY",
    "<author_addiction>": "ADDICTION",
}

LABELS = {
    str(k).strip().strip("<>").strip(): v
    for k, v in PLACEHOLDER_TO_LABEL.items()
}


# Converts the content of the substitutions_dictionary column into a real Python dictionary.
def parse_substitutions(value):
    if pd.isna(value):
        return {}

    if isinstance(value, dict):
        return value

    value = str(value).strip()

    if not value or value.lower() in ("nan", "none", "{}"):
        return {}

    try:
        result = ast.literal_eval(value)
        if isinstance(result, dict):
            return result
    except Exception:
        pass

    try:
        result = json.loads(value)
        if isinstance(result, dict):
            return result
    except Exception:
        pass

    try:
        value = (
            value.replace("\u2018", "'")
                 .replace("\u2019", "'")
                 .replace("\u201c", '"')
                 .replace("\u201d", '"')
        )

        if value.startswith("{") and "'" in value and '"' not in value:
            value = value.replace("'", '"')

        result = json.loads(value)
        return result if isinstance(result, dict) else {}
    except Exception:
        return {}


# Load models (NER + LLM)
def load_models():
    nlp = spacy.load(NER_MODEL_PATH)

    tokenizer = AutoTokenizer.from_pretrained(LLM_BASE_CHECKPOINT)

    model = T5ForSequenceClassification.from_pretrained(
        LLM_BASE_CHECKPOINT,
        num_labels=2
    )

    state_dict = torch.load(
        LLM_WEIGHTS_PATH,
        map_location=DEVICE
    )

    if isinstance(state_dict, dict) and "model_state_dict" in state_dict:
        state_dict = state_dict["model_state_dict"]

    model.load_state_dict(state_dict, strict=False)
    model.to(DEVICE)
    model.eval()

    return nlp, tokenizer, model


# Count the sentences containing PHI in the input CSV (ground_truth).
def has_phi_ground_truth(value):
    return any(
        str(k).strip().strip("<>").strip() in LABELS
        for k in parse_substitutions(value)
    )


# LLM prediction
@torch.no_grad()
def predict_phi(texts, tokenizer, model):
    inputs = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=MAX_INPUT_LENGTH,
        return_tensors="pt"
    ).to(DEVICE)

    preds = model(**inputs).logits.argmax(dim=-1)

    return [
        "Yes" if pred.item() == POSITIVE_CLASS_INDEX else "No"
        for pred in preds
    ]


# Finds substitution values in the text and assigns BIO labels to each token.
def tokenize_and_label(text, substitutions):
    char_labels = ["O"] * len(text)
    text_lower = text.lower()

    for placeholder, value in substitutions.items():
        label = LABELS.get(
            str(placeholder).strip().strip("<>").strip()
        )

        if not label:
            continue

        value = str(value)

        if not value.strip():
            continue

        value_lower = value.lower()
        start = 0

        while True:
            start = text_lower.find(value_lower, start)

            if start < 0:
                break

            for i in range(start, start + len(value)):
                char_labels[i] = "I-" + label

            char_labels[start] = "B-" + label
            start += len(value)

    tokens = []
    tags = []

    for match in re.finditer(r"\S+", text):
        token = match.group()
        start = match.start()
        tag = char_labels[start]

        if tag == "O":
            for i in range(start, match.end()):
                if char_labels[i].startswith("B-"):
                    tag = char_labels[i]
                    break

        if tag.startswith("I-") and (
            start == 0 or char_labels[start - 1] != tag
        ):
            tag = "B-" + tag.split("-", 1)[1]

        if len(token) > 1 and token[-1] in "?.!:;,()[]'":
            token = token[:-1]

        if token:
            tokens.append(token)
            tags.append(tag)

    return tokens, tags


# Build datasets to feed into the NER system
def build_ner_dataset(df):
    records = []

    for _, row in df.iterrows():
        text = row[TEXT_COLUMN]

        if not isinstance(text, str):
            continue

        tokens, tags = tokenize_and_label(
            text,
            parse_substitutions(row[SUBS_COLUMN])
        )

        records.append({
            "tokens": tokens,
            "ent_tags": tags
        })

    return records


# Convert the dataset prepared for NER into the Example format required by spaCy to evaluate the NER model.
def create_examples(records, nlp):
    examples = []

    for item in records:
        tokens = item["tokens"]
        tags = item["ent_tags"]
        text = " ".join(tokens)

        entities = []
        position = 0

        for token, tag in zip(tokens, tags):
            end = position + len(token)

            if tag != "O":
                entities.append((position, end, tag))

            position = end + 1

        example = Example.from_dict(
            nlp.make_doc(text),
            {"entities": entities}
        )

        example.predicted = nlp(text)
        examples.append(example)

    return examples


def main():
    nlp, tokenizer, llm_model = load_models()
    df = pd.read_csv(CSV_PATH)

    if TEXT_COLUMN not in df.columns or SUBS_COLUMN not in df.columns:
        raise ValueError("Colonne richieste non presenti nel CSV.")

    df["has_phi_gt"] = df[SUBS_COLUMN].apply(has_phi_ground_truth)

    texts = df[TEXT_COLUMN].fillna("").astype(str).tolist()
    predictions = []

    for i in range(0, len(texts), LLM_BATCH_SIZE):
        predictions.extend(
            predict_phi(
                texts[i:i + LLM_BATCH_SIZE],
                tokenizer,
                llm_model
            )
        )

    df["llm_pred"] = predictions

    df_phi = df[df["llm_pred"] == "Yes"].reset_index(drop=True)
    df_nophi = df[df["llm_pred"] == "No"].reset_index(drop=True)

    llm_fn = df_nophi[df_nophi["has_phi_gt"]]

    ner_records = build_ner_dataset(df_phi)
    examples = create_examples(ner_records, nlp)

    ner_pred = [
        "Yes" if example.predicted.ents else "No"
        for example in examples
    ]

    df_phi["ner_pred"] = ner_pred

    llm_false_positives = df_phi[~df_phi["has_phi_gt"]]

    fp_eliminated = (
        llm_false_positives["ner_pred"] == "No"
    ).sum()

    print(f"\nLLM FN trovati: {len(llm_fn)}")

    print(
        f"LLM FP: "
        f"{len(llm_false_positives)}"
    )

    print(
        f"LLM FP eliminati dal NER: "
        f"{fp_eliminated}/{len(llm_false_positives)}"
    )

if __name__ == "__main__":
    main()